<a href="https://colab.research.google.com/github/AshokGit544/Document-RAG-Assistant1/blob/main/Document_RAG_Assistant1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install pandas scikit-learn pypdf python-docx gradio

import os
import re
import json
import zipfile
from pathlib import Path

import pandas as pd
from pypdf import PdfReader
from docx import Document
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from google.colab import files

PROJECT_DIR = Path("Document-RAG-Assistant")
PROJECT_DIR.mkdir(exist_ok=True)

print("Upload your file now...")
uploaded = files.upload()

uploaded_files = list(uploaded.keys())
print("Uploaded files:", uploaded_files)

file_path = uploaded_files[0]
print("Using file:", file_path)

def load_pdf(path):
    reader = PdfReader(path)
    text = []
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text.append(page_text)
    return "\n".join(text).strip()

def load_docx(path):
    doc = Document(path)
    return "\n".join([p.text for p in doc.paragraphs if p.text.strip()]).strip()

def load_txt(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read().strip()

def load_document(path):
    lower = path.lower()
    if lower.endswith(".pdf"):
        return load_pdf(path)
    elif lower.endswith(".docx"):
        return load_docx(path)
    elif lower.endswith(".txt"):
        return load_txt(path)
    else:
        raise ValueError("Only PDF, DOCX, and TXT files are supported.")

raw_text = load_document(file_path)

print("\nDocument loaded successfully")
print("Total characters:", len(raw_text))

def clean_text(text):
    text = text.replace("\x00", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

cleaned_text = clean_text(raw_text)

def chunk_text(text, chunk_size=900, overlap=150):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

chunks = chunk_text(cleaned_text, chunk_size=900, overlap=150)

chunks_df = pd.DataFrame({
    "chunk_id": [f"CHUNK_{i+1:03d}" for i in range(len(chunks))],
    "chunk_text": chunks
})

chunks_df.to_csv(PROJECT_DIR / "document_chunks.csv", index=False)

print("\nChunks created:", len(chunks))
display(chunks_df.head())

vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = vectorizer.fit_transform(chunks_df["chunk_text"])

def retrieve_chunks(query, top_k=5):
    query_vec = vectorizer.transform([query])
    sims = cosine_similarity(query_vec, tfidf_matrix)[0]
    top_idx = sims.argsort()[::-1][:top_k]

    result = chunks_df.iloc[top_idx].copy()
    result["similarity_score"] = sims[top_idx]
    return result.reset_index(drop=True)

def extractive_answer(query, top_k=5):
    retrieved = retrieve_chunks(query, top_k=top_k)

    combined = " ".join(retrieved["chunk_text"].tolist())
    sentences = re.split(r'(?<=[.!?])\s+', combined)

    query_terms = set(re.findall(r'\b\w+\b', query.lower()))
    scored_sentences = []

    for sentence in sentences:
        sentence_terms = set(re.findall(r'\b\w+\b', sentence.lower()))
        score = len(query_terms.intersection(sentence_terms))
        if len(sentence.strip()) > 30:
            scored_sentences.append((sentence, score))

    scored_sentences = sorted(scored_sentences, key=lambda x: x[1], reverse=True)
    best_sentences = [s[0] for s in scored_sentences[:4] if s[1] > 0]

    if best_sentences:
        answer = " ".join(best_sentences)
    else:
        answer = "The answer is not clearly available in the uploaded document."

    return answer, retrieved

sample_questions = [
    "What does the document say about AI governance?",
    "What systems are mentioned in the enterprise architecture?",
    "What are the future roadmap priorities?",
    "How does the organization handle security?",
    "What does the document say about digital transformation?"
]

all_results = []

for q in sample_questions:
    answer, retrieved = extractive_answer(q, top_k=5)
    print("\n" + "="*100)
    print("QUESTION:", q)
    print("\nANSWER:")
    print(answer)
    print("\nTOP RETRIEVED CHUNKS:")
    display(retrieved)
    all_results.append({
        "question": q,
        "answer": answer
    })

results_df = pd.DataFrame(all_results)
results_df.to_csv(PROJECT_DIR / "sample_query_answers.csv", index=False)

readme_text = """# Document RAG Assistant

This project is a simple document question answering system.

It takes a document, splits it into chunks, converts the chunks into searchable vectors, and retrieves the most relevant chunks for a user question. Then it generates a grounded answer using only the retrieved document content.

## What this project does

- uploads a document
- reads PDF, DOCX, or TXT files
- cleans the text
- splits the document into chunks
- creates TF-IDF vectors
- retrieves the most relevant chunks for a question
- generates a grounded answer from retrieved text
- saves output files

## Main files

- `document_chunks.csv`
- `sample_query_answers.csv`
- `app.py`
- `requirements.txt`
- `README.md`

## Tools used

- Python
- pandas
- scikit-learn
- pypdf
- python-docx
- gradio

## How to run

1. Open the notebook in Google Colab
2. Upload your document
3. Run all cells
4. Ask questions
5. Review the saved output files
"""

requirements_text = """pandas
scikit-learn
pypdf
python-docx
gradio
"""

gitignore_text = """__pycache__/
.ipynb_checkpoints/
*.pyc
.DS_Store
"""

app_code = r'''
import re
import pandas as pd
import gradio as gr
from pypdf import PdfReader
from docx import Document
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def load_pdf(file_obj):
    reader = PdfReader(file_obj.name)
    text = []
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text.append(page_text)
    return "\n".join(text).strip()

def load_docx(file_obj):
    doc = Document(file_obj.name)
    return "\n".join([p.text for p in doc.paragraphs if p.text.strip()]).strip()

def load_txt(file_obj):
    with open(file_obj.name, "r", encoding="utf-8", errors="ignore") as f:
        return f.read().strip()

def load_document(file_obj):
    name = file_obj.name.lower()
    if name.endswith(".pdf"):
        return load_pdf(file_obj)
    elif name.endswith(".docx"):
        return load_docx(file_obj)
    elif name.endswith(".txt"):
        return load_txt(file_obj)
    else:
        raise ValueError("Only PDF, DOCX, and TXT files are supported.")

def clean_text(text):
    text = text.replace("\x00", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def chunk_text(text, chunk_size=900, overlap=150):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

def answer_question(file_obj, question, top_k):
    if file_obj is None:
        return "Please upload a document first.", ""

    raw_text = load_document(file_obj)
    cleaned_text = clean_text(raw_text)
    chunks = chunk_text(cleaned_text)

    chunks_df = pd.DataFrame({
        "chunk_id": [f"CHUNK_{i+1:03d}" for i in range(len(chunks))],
        "chunk_text": chunks
    })

    vectorizer = TfidfVectorizer(stop_words="english")
    tfidf_matrix = vectorizer.fit_transform(chunks_df["chunk_text"])

    query_vec = vectorizer.transform([question])
    sims = cosine_similarity(query_vec, tfidf_matrix)[0]
    top_idx = sims.argsort()[::-1][:top_k]

    retrieved = chunks_df.iloc[top_idx].copy()
    retrieved["similarity_score"] = sims[top_idx]

    combined = " ".join(retrieved["chunk_text"].tolist())
    sentences = re.split(r'(?<=[.!?])\s+', combined)

    query_terms = set(re.findall(r'\b\w+\b', question.lower()))
    scored_sentences = []

    for sentence in sentences:
        sentence_terms = set(re.findall(r'\b\w+\b', sentence.lower()))
        score = len(query_terms.intersection(sentence_terms))
        if len(sentence.strip()) > 30:
            scored_sentences.append((sentence, score))

    scored_sentences = sorted(scored_sentences, key=lambda x: x[1], reverse=True)
    best_sentences = [s[0] for s in scored_sentences[:4] if s[1] > 0]

    if best_sentences:
        answer = " ".join(best_sentences)
    else:
        answer = "The answer is not clearly available in the uploaded document."

    retrieved_text = retrieved[["chunk_id", "similarity_score", "chunk_text"]].to_string(index=False)
    return answer, retrieved_text

demo = gr.Interface(
    fn=answer_question,
    inputs=[
        gr.File(label="Upload Document"),
        gr.Textbox(label="Question", lines=2, placeholder="Ask a question about the document"),
        gr.Slider(1, 10, value=5, step=1, label="Top K Chunks")
    ],
    outputs=[
        gr.Textbox(label="Answer"),
        gr.Textbox(label="Retrieved Chunks")
    ],
    title="Document RAG Assistant",
    description="Upload a PDF, DOCX, or TXT file and ask grounded questions from the document."
)

if __name__ == "__main__":
    demo.launch()
'''

(PROJECT_DIR / "README.md").write_text(readme_text, encoding="utf-8")
(PROJECT_DIR / "requirements.txt").write_text(requirements_text, encoding="utf-8")
(PROJECT_DIR / ".gitignore").write_text(gitignore_text, encoding="utf-8")
(PROJECT_DIR / "app.py").write_text(app_code, encoding="utf-8")

zip_path = Path("Document-RAG-Assistant.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for file_path in PROJECT_DIR.iterdir():
        zf.write(file_path, arcname=file_path.name)

print("\nProject files created:")
for p in sorted(PROJECT_DIR.iterdir()):
    print("-", p.name)

print("\nZIP file created:", zip_path.resolve())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.2/332.2 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 5.0 MB/s eta 0:00:00
Upload your file now...


Saving LLM_Practice_Document_Sample.pdf to LLM_Practice_Document_Sample.pdf
Uploaded files: ['LLM_Practice_Document_Sample.pdf']
Using file: LLM_Practice_Document_Sample.pdf

Document loaded successfully
Total characters: 270649

Chunks created: 361


,chunk_id,chunk_text
0,CHUNK_001,Enterprise AI Transformation – Sample Practice...
1,CHUNK_002,entralized data warehouse along with distribut...
2,CHUNK_003,ies. The security model includes identity and ...
3,CHUNK_004,ntaining competitive advantage and adapting to...
4,CHUNK_005,centralized data warehouse along with distribu...



QUESTION: What does the document say about AI governance?

ANSWER:
Enterprise AI Transformation – Sample Practice Document Company Overview This organization operates in a highly competitive global market and focuses on delivering innovative technology-driven solutions. The future roadmap prioritizes scalable AI systems, enhanced automation pipelines, ethical AI p . The future roadmap prioritizes scalable AI systems, enhanced automation pipelines, ethical AI practices, and s. The future roadmap prioritizes scalable AI systems, enhanced automation pipelines, ethical AI practices, and workforce upskilling initiatives.

TOP RETRIEVED CHUNKS:


,chunk_id,chunk_text,similarity_score
0,CHUNK_001,Enterprise AI Transformation – Sample Practice...,0.265199
1,CHUNK_180,and monitoring. Cross-functional collaboration...,0.082216
2,CHUNK_159,. Cross-functional collaboration ensures align...,0.082129
3,CHUNK_099,s. Governance policies define strict access co...,0.082008
4,CHUNK_120,n business stakeholders and technical teams. G...,0.081984


...


QUESTION: How does the organization handle security?

ANSWER:
The organization follows a structured lifecycle including problem identification, data collection, model training, evaluation, deployment, and monitoring. The security model includes identity and access management, intrusion detection systems, endpoint protection, and continuous vulnerability assessments. The organization follows a structured lifecycle including problem identification, data collection, model training, evaluation, deployment, and monitoring. The security model includes identity and access management, intrusion detection systems, endpoint protection, and continuous vulnerability assessments.

TOP RETRIEVED CHUNKS:


,chunk_id,chunk_text,similarity_score
0,CHUNK_294,"g, recommendation engines, and intelligent aut...",0.178145
1,CHUNK_309,"processing, recommendation engines, and intell...",0.178079
2,CHUNK_354,"e initiatives include predictive analytics, na...",0.177306
3,CHUNK_039,"s, and compliance frameworks aligned with regu...",0.177005
4,CHUNK_333,"include predictive analytics, natural language...",0.175670



QUESTION: What does the document say about digital transformation?

ANSWER:
Enterprise AI Transformation – Sample Practice Document Company Overview This organization operates in a highly competitive global market and focuses on delivering innovative technology-driven solutions. Over the past decade, leadership has invested heavily in digital systems, automation platforms, and enterprise data architecture to ensure scalable growth and measurable business outcomes. Digital Transformation Strategy This organization operates in a highly competitive global market and focuses on delivering innovative technology-driven solutions. Over the past decade, leadership has invested heavily in digital systems, automation platforms, and enterprise data architecture to ensure scalable growth and measurable business outcomes.

TOP RETRIEVED CHUNKS:


,chunk_id,chunk_text,similarity_score
0,CHUNK_001,Enterprise AI Transformation – Sample Practice...,0.364146
1,CHUNK_037,remains central to maintaining competitive adv...,0.221691
2,CHUNK_036,ed with regulatory bodies. The security model ...,0.194657
3,CHUNK_280,", and automated service systems. Feedback loop...",0.023007
4,CHUNK_307,"including web platforms, mobile applications, ...",0.022953



Project files created:
- .gitignore
- README.md
- app.py
- document_chunks.csv
- requirements.txt
- sample_query_answers.csv

ZIP file created: /content/Document-RAG-Assistant.zip
